# Train Specialized ToM Steering Vectors with Persona Templates

This notebook trains 6 specialized Theory of Mind steering vectors using:
- Persona templates from `haiku_plan.md`
- 750 procedural examples per vector (from stories_train.csv)
- 582 full truncated output examples
- Chat template format with system/user/assistant structure
- Training on ALL layers

**Vectors:**
1. core_tom - Mixed data from all 0_ conditions
2. forward_belief_true - From 0_forward_belief_true_belief
3. forward_belief_false - From 0_forward_belief_false_belief
4. backward_belief - From 0_backward_belief (both true/false)
5. forward_action_true - From 0_forward_action_true_belief
6. forward_action_false - From 0_forward_action_false_belief

**Model:** google/gemma-3-4b-it

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the Cogni_map repository (if running in Colab)
import os
if not os.path.exists('Cogni_map'):
    !git clone https://github.com/ChuloIva/Cogni_map.git
    %cd Cogni_map/ToM
else:
    print("Repository already cloned")

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate sentencepiece
!pip install -q hatchling
!pip install -q repeng/ 2>/dev/null || pip install -q ToM/repeng/

# Verify installation
import sys
if 'repeng' not in sys.path:
    sys.path.insert(0, 'repeng')

try:
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng successfully imported!")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    !pip install -q numpy>=1.26.4 scikit-learn>=1.4.0 tqdm>=4.66.1 gguf>=0.13.0
    from repeng import ControlVector, ControlModel, DatasetEntry
    print("✓ repeng imported!")

In [ ]:
# Optional: Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/tom_steering_vectors
    print("✓ Google Drive mounted")
except:
    print("Not running in Colab, skipping Drive mount")

## 2. Define Persona Templates

These are the persona templates from `haiku_plan.md` that will be used as system messages.

In [ ]:
# Persona templates from haiku_plan.md
PERSONA_TEMPLATES = {
    "core_tom": {
        "name": "Core Theory of Mind",
        "system_instruction": "Act as if you have {persona}.",
        "positive": [
            "an exceptional ability to take others' perspectives and simulate their mental states",
            "a strong capacity for counterfactual reasoning about what others believe to be true",
            "a remarkable talent for understanding how multiple agents' beliefs interact and diverge"
        ],
        "negative": [
            "a very limited ability to understand how others think and feel",
            "a poor capacity for inhibiting your own knowledge to understand another's perspective",
            "a fundamental confusion about the difference between your own knowledge and what others know"
        ]
    },
    
    "forward_belief_true": {
        "name": "Forward Belief - True Belief",
        "system_instruction": "You track what people believe as events happen {persona}.",
        "positive": [
            "with high accuracy, updating their beliefs based on new information they receive",
            "exceptionally well, recognizing when an agent has observed information and therefore holds an accurate belief",
            "precisely, understanding that when people see events, their beliefs align with reality"
        ],
        "negative": [
            "poorly, struggling to understand that a person's knowledge is updated by what they observe",
            "incorrectly, failing to recognize that seeing something guarantees knowledge of it",
            "inaccurately, often doubting beliefs that are actually grounded in direct observation"
        ]
    },
    
    "forward_belief_false": {
        "name": "Forward Belief - False Belief",
        "system_instruction": "You track what people believe as events happen {persona}.",
        "positive": [
            "exceptionally well, recognizing when an agent has missed information and therefore holds a false belief",
            "with excellence, understanding that people can believe things that contradict reality if they lack key information",
            "skillfully, recognizing the difference between what happened and what the agent thinks happened"
        ],
        "negative": [
            "poorly, struggling to understand that a person's knowledge is limited to what they have observed",
            "inaccurately, always assuming agents know the current truth regardless of what they've seen",
            "incorrectly, failing to grasp that unawareness creates false beliefs"
        ]
    },
    
    "backward_belief": {
        "name": "Backward Belief (Abductive Reasoning)",
        "system_instruction": "You infer past beliefs from current evidence {persona}.",
        "positive": [
            "skillfully, reconstructing what someone must have believed by looking at their later actions",
            "with excellence, determining if an outcome was due to a prior false belief or a change in the world",
            "expertly, working backwards from behavior to identify the underlying mental state that caused it"
        ],
        "negative": [
            "poorly, unable to work backward from an action to understand the belief that caused it",
            "inaccurately, failing to distinguish between ignorance and a changed reality when explaining events",
            "confusingly, often confusing what the person knew with what they did"
        ]
    },
    
    "forward_action_true": {
        "name": "Forward Action - True Belief",
        "system_instruction": "At predicting what people will do, you are {persona}.",
        "positive": [
            "able to simulate the plan an agent will follow based on their true beliefs about the world, with excellent accuracy",
            "especially adept at predicting actions when an agent's beliefs accurately reflect reality",
            "skilled at understanding that accurate beliefs lead to rational, goal-aligned actions"
        ],
        "negative": [
            "unable to predict actions based on how agents with accurate beliefs will behave, performing poorly",
            "struggling to connect true beliefs with the logical actions they produce, showing a lack of skill",
            "often confused, predicting actions that contradict what someone with accurate beliefs would do"
        ]
    },
    
    "forward_action_false": {
        "name": "Forward Action - False Belief",
        "system_instruction": "At predicting what people will do, you are {persona}.",
        "positive": [
            "able to simulate the plan an agent will follow based on their potentially false beliefs about the world, with excellent insight",
            "especially adept at predicting actions that logically follow from an agent's mistaken beliefs",
            "skilled at understanding that even false beliefs drive coherent, goal-directed behavior"
        ],
        "negative": [
            "able to predict actions only based on the actual state of the world while ignoring individual beliefs, performing poorly",
            "assuming that people always act with perfect and complete information, showing a lack of skill",
            "confused, predicting what someone should do instead of what their false beliefs would lead them to do"
        ]
    }
}

print(f"Loaded {len(PERSONA_TEMPLATES)} persona templates:")
for key, template in PERSONA_TEMPLATES.items():
    print(f"  - {key}: {template['name']}")

## 3. Load Model (Gemma-3-4B)

In [ ]:
import json
import torch
import sys
from transformers import AutoModelForCausalLM, AutoConfig, Gemma3ForCausalLM, AutoTokenizer

# Ensure repeng is in path
if 'repeng' not in sys.path:
    sys.path.insert(0, 'repeng')

from repeng import ControlVector, ControlModel, DatasetEntry

print("✓ All imports successful!")

In [ ]:
# Model configuration
model_name = "google/gemma-3-4b-it"

print(f"Loading {model_name}...")

# Load config
config = AutoConfig.from_pretrained(model_name)

# Use bfloat16 for better numerical stability
print("Using bfloat16 for better numerical stability...")

# Load model
if hasattr(config, 'vision_config'):
    print("Detected vision-language model. Loading text-only version...")
    base_model = Gemma3ForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
else:
    print("Loading standard causal LM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

print("Model loaded successfully!")
print(f"Device: {base_model.device}")
print(f"Dtype: {base_model.dtype}")

In [ ]:
# Wrap with ControlModel for steering - USING ALL LAYERS
print("Setting up ControlModel...")

# For multimodal Gemma3, layers are at model.language_model.layers
if hasattr(base_model, 'language_model') and hasattr(base_model.language_model, 'layers'):
    print(f"Detected multimodal Gemma3 architecture")
    base_model.repeng_layers = base_model.language_model.layers
    num_layers = len(base_model.language_model.layers)
    base_model.config.num_hidden_layers = num_layers
elif hasattr(base_model, 'model') and hasattr(base_model.model, 'layers'):
    print(f"Detected standard architecture")
    base_model.repeng_layers = base_model.model.layers
    num_layers = len(base_model.model.layers)
else:
    raise ValueError("Could not find model layers!")

print(f"Total layers: {num_layers}")


# Using all layers except the first (layers 1-32)
layer_ids = list(range(-1, -32, -1))
print(f"Wrapping ALL layers: 0 to {num_layers-1}")

model = ControlModel(base_model, layer_ids)

print(f"✓ ControlModel initialized successfully!")
print(f"Training will use all {num_layers} layers")

## 4. Load Training Data

In [ ]:
import csv
from pathlib import Path
from typing import List, Dict, Tuple

def load_procedural_data(condition_path: Path) -> List[Tuple[str, str, str, str]]:
    """
    Load procedural training data from stories_train.csv.
    
    Returns:
        List of (story, question, correct_answer, incorrect_answer) tuples
    """
    train_csv = condition_path / 'stories_train.csv'
    
    data = []
    with open(train_csv, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=';')
        for row in reader:
            if len(row) == 4:
                story, question, correct, incorrect = row
                data.append((story, question, correct, incorrect))
    
    return data


# Define condition paths
conditions_base = Path('procedural-evals-tom/data/conditions')

CONDITION_PATHS = {
    "forward_belief_true": conditions_base / "0_forward_belief_true_belief",
    "forward_belief_false": conditions_base / "0_forward_belief_false_belief",
    "backward_belief_true": conditions_base / "0_backward_belief_true_belief",
    "backward_belief_false": conditions_base / "0_backward_belief_false_belief",
    "forward_action_true": conditions_base / "0_forward_action_true_belief",
    "forward_action_false": conditions_base / "0_forward_action_false_belief",
}

# Load all condition data
print("Loading procedural training data...")
procedural_data = {}
for key, path in CONDITION_PATHS.items():
    data = load_procedural_data(path)
    procedural_data[key] = data
    print(f"  {key}: {len(data)} examples")

print(f"\nTotal procedural examples: {sum(len(d) for d in procedural_data.values())}")

In [ ]:
# Load truncated outputs (full, not truncated)
truncated_outputs_path = Path('repeng/notebooks/data/all_truncated_outputs.json')

with open(truncated_outputs_path, 'r', encoding='utf-8') as f:
    truncated_outputs = json.load(f)

print(f"Loaded {len(truncated_outputs)} truncated output suffixes")
print(f"Examples: {truncated_outputs[:5]}")

## 5. Create Dataset Builder Functions

In [ ]:
def create_chat_message(system_prompt: str, user_message: str, assistant_message: str, tokenizer) -> str:
    """
    Create a properly formatted chat message using the model's chat template.
    
    Note: Gemma doesn't support system messages in the standard way, so we'll
    prepend the system instruction to the user message.
    """
    # Combine system instruction with user message
    combined_user_message = f"{system_prompt}\n\n{user_message}"
    
    messages = [
        {"role": "user", "content": combined_user_message}
    ]
    
    # Apply chat template and add assistant response
    formatted = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    
    # Add assistant's response
    return formatted + assistant_message


def create_procedural_dataset(
    persona_template: Dict,
    procedural_examples: List[Tuple[str, str, str, str]],
    tokenizer
) -> List[DatasetEntry]:
    """
    Create dataset from procedural examples using persona template.
    
    Args:
        persona_template: Dict with 'system_instruction', 'positive', 'negative'
        procedural_examples: List of (story, question, correct, incorrect) tuples
        tokenizer: Tokenizer for chat template
    
    Returns:
        List of DatasetEntry objects
    """
    dataset = []
    
    system_template = persona_template['system_instruction']
    positive_personas = persona_template['positive']
    negative_personas = persona_template['negative']
    
    for story, question, correct_answer, incorrect_answer in procedural_examples:
        # Format user message: story + question
        user_message = f"{story}\n\n{question}"
        
        # Create entries for each persona pair
        for pos_persona, neg_persona in zip(positive_personas, negative_personas):
            pos_system = system_template.format(persona=pos_persona)
            neg_system = system_template.format(persona=neg_persona)
            
            # Positive example: good persona + correct answer
            positive_text = create_chat_message(
                pos_system, user_message, correct_answer, tokenizer
            )
            
            # Negative example: bad persona + incorrect answer
            negative_text = create_chat_message(
                neg_system, user_message, incorrect_answer, tokenizer
            )
            
            dataset.append(DatasetEntry(
                positive=positive_text,
                negative=negative_text
            ))
    
    return dataset


def create_truncated_dataset(
    persona_template: Dict,
    truncated_outputs: List[str],
    tokenizer
) -> List[DatasetEntry]:
    """
    Create dataset from truncated outputs using persona template.
    
    Args:
        persona_template: Dict with 'system_instruction', 'positive', 'negative'
        truncated_outputs: List of output suffixes (used as-is, not truncated)
        tokenizer: Tokenizer for chat template
    
    Returns:
        List of DatasetEntry objects
    """
    dataset = []
    
    system_template = persona_template['system_instruction']
    positive_personas = persona_template['positive']
    negative_personas = persona_template['negative']
    
    # Use each truncated output with each persona pair
    for suffix in truncated_outputs:
        for pos_persona, neg_persona in zip(positive_personas, negative_personas):
            pos_system = system_template.format(persona=pos_persona)
            neg_system = system_template.format(persona=neg_persona)
            
            # Create entries with just the persona instruction + suffix
            # (no specific user message, just the system instruction)
            messages_pos = [{"role": "user", "content": pos_system}]
            messages_neg = [{"role": "user", "content": neg_system}]
            
            pos_formatted = tokenizer.apply_chat_template(
                messages_pos, add_generation_prompt=True, tokenize=False
            )
            neg_formatted = tokenizer.apply_chat_template(
                messages_neg, add_generation_prompt=True, tokenize=False
            )
            
            dataset.append(DatasetEntry(
                positive=pos_formatted + suffix,
                negative=neg_formatted + suffix
            ))
    
    return dataset


def create_full_dataset(
    vector_name: str,
    persona_template: Dict,
    procedural_data_dict: Dict[str, List],
    truncated_outputs: List[str],
    tokenizer
) -> List[DatasetEntry]:
    """
    Create complete dataset for a vector by combining procedural and truncated data.
    
    Args:
        vector_name: Name of vector (e.g., 'forward_belief_true')
        persona_template: Persona template dict
        procedural_data_dict: Dict mapping condition names to data
        truncated_outputs: List of truncated output suffixes
        tokenizer: Tokenizer
    
    Returns:
        Combined dataset
    """
    print(f"\nCreating dataset for {vector_name}...")
    
    # Select appropriate procedural data based on vector type
    if vector_name == "core_tom":
        # Mix all 0_ conditions
        procedural_examples = []
        for key in procedural_data_dict:
            procedural_examples.extend(procedural_data_dict[key])
        print(f"  Using mixed data from all conditions: {len(procedural_examples)} examples")
    elif vector_name == "backward_belief":
        # Combine both true and false
        procedural_examples = (
            procedural_data_dict["backward_belief_true"] +
            procedural_data_dict["backward_belief_false"]
        )
        print(f"  Using backward_belief (true+false): {len(procedural_examples)} examples")
    else:
        # Use specific condition
        procedural_examples = procedural_data_dict[vector_name]
        print(f"  Using {vector_name}: {len(procedural_examples)} examples")
    
    # Create datasets
    print("  Creating procedural dataset...")
    procedural_dataset = create_procedural_dataset(
        persona_template, procedural_examples, tokenizer
    )
    
    print("  Creating truncated outputs dataset...")
    truncated_dataset = create_truncated_dataset(
        persona_template, truncated_outputs, tokenizer
    )
    
    # Combine
    full_dataset = procedural_dataset + truncated_dataset
    
    print(f"  Total dataset size: {len(full_dataset)} pairs")
    print(f"    Procedural: {len(procedural_dataset)}")
    print(f"    Truncated: {len(truncated_dataset)}")
    
    return full_dataset

print("✓ Dataset builder functions defined")

## 6. Train Vectors

Train each of the 6 specialized vectors.

In [ ]:
# Test dataset creation with one example first
print("Testing dataset creation...\n")

test_vector = "forward_belief_true"
test_template = PERSONA_TEMPLATES[test_vector]

test_dataset = create_full_dataset(
    test_vector,
    test_template,
    procedural_data,
    truncated_outputs,
    tokenizer
)

print(f"\n✓ Test dataset created successfully!")
print(f"\nExample training pair:")
print(f"Positive: {test_dataset[0].positive[:300]}...")
print(f"\nNegative: {test_dataset[0].negative[:300]}...")

In [ ]:
# Train all 6 vectors
import os

# Create output directory
output_dir = Path('steering_vectors')
output_dir.mkdir(exist_ok=True)

trained_vectors = {}

for vector_name, persona_template in PERSONA_TEMPLATES.items():
    print("\n" + "="*80)
    print(f"Training: {vector_name} - {persona_template['name']}")
    print("="*80)
    
    # Create dataset
    dataset = create_full_dataset(
        vector_name,
        persona_template,
        procedural_data,
        truncated_outputs,
        tokenizer
    )
    
    # Train vector
    print(f"\nTraining vector (this may take several minutes)...")
    model.reset()  # Always reset before training
    
    vector = ControlVector.train(
        model,
        tokenizer,
        dataset,
        method='pca_center'
    )
    
    print(f"\n✓ Training complete!")
    print(f"  Vector contains directions for {len(vector.directions)} layers")
    
    # Export
    output_path = output_dir / f"tom_{vector_name}_persona_all_layers.gguf"
    vector.export_gguf(str(output_path))
    print(f"  ✓ Exported to: {output_path}")
    
    # Store for later use
    trained_vectors[vector_name] = vector
    
    # Try to save to Google Drive if available
    try:
        import shutil
        drive_path = f"/content/drive/MyDrive/tom_steering_vectors/{output_path.name}"
        shutil.copy(str(output_path), drive_path)
        print(f"  ✓ Also saved to Google Drive")
    except:
        pass

print("\n" + "="*80)
print("ALL VECTORS TRAINED SUCCESSFULLY!")
print("="*80)
print(f"\nTrained {len(trained_vectors)} vectors:")
for name in trained_vectors.keys():
    print(f"  ✓ {name}")

## 7. Test Vectors

Quick test to verify the vectors work as expected.

In [ ]:
def generate_text(prompt, model, tokenizer, max_new_tokens=128):
    """Generate text from the model."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Test with a classic false belief scenario
test_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy?

Answer:"""

print("Testing forward_belief_false vector...\n")
print("="*80)

# Baseline
print("\n[BASELINE - No Steering]")
model.reset()
baseline = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(baseline)

# With positive steering
print("\n" + "="*80)
print("\n[WITH FORWARD_BELIEF_FALSE STEERING - Strength: 1.5]")
print("Expected: Should recognize Sarah will look in red box")
print("-"*80)
model.set_control(trained_vectors['forward_belief_false'], coeff=1.5)
steered = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(steered)

model.reset()
print("\n" + "="*80)

## 8. Summary

All 6 specialized ToM vectors have been trained and exported:

1. **tom_core_tom_persona_all_layers.gguf** - General ToM capabilities
2. **tom_forward_belief_true_persona_all_layers.gguf** - Tracking true beliefs
3. **tom_forward_belief_false_persona_all_layers.gguf** - Tracking false beliefs
4. **tom_backward_belief_persona_all_layers.gguf** - Abductive reasoning
5. **tom_forward_action_true_persona_all_layers.gguf** - Predicting actions from true beliefs
6. **tom_forward_action_false_persona_all_layers.gguf** - Predicting actions from false beliefs

### Training Details:
- Model: Gemma-3-4b-it
- Layers: ALL layers (0 to 41)
- Data per vector: ~1,332 pairs (procedural + truncated)
- Format: Chat template with system/user/assistant structure
- Method: PCA centering

### Usage:
```python
from repeng import ControlVector, ControlModel

# Load vector
vector = ControlVector.import_gguf("tom_forward_belief_false_persona_all_layers.gguf")

# Apply steering
model.set_control(vector, coeff=1.5)

# Generate...

# Reset
model.reset()
```